# Predicting the likelihood of marketing engagement with Python

In [ ]:
# Load the dataset
import pandas as pd
import janitor
df = pd.read_csv('../../data/raw/WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv').clean_names(case_type='snake')

In [ ]:
df.shape

In [ ]:
df.head()

# Variable Encoding

## Response Variable: response

In [ ]:
df['engaged'] = df['response'].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
df['engaged'].mean()

## Features

In [ ]:
df.describe()

In [ ]:
continuous_features = [
    'customer_lifetime_value', 'income', 'monthly_premium_auto',
    'months_since_last_claim', 'months_since_policy_inception',
    'number_of_open_complaints', 'number_of_policies', 'total_claim_amount'
]

#### Creating Dummy Variables

In [ ]:
columns_to_encode = [
    'sales_channel', 'vehicle_size', 'vehicle_class', 'policy', 'policy_type', 
    'employment_status', 'marital_status', 'education', 'coverage'
]

categorical_features = []
for col in columns_to_encode:
    encoded_df = pd.get_dummies(df[col])
    encoded_df.columns = [col.replace(' ', '.') + '.' + x for x in encoded_df.columns]
    
    categorical_features += list(encoded_df.columns)
    
    df = pd.concat([df, encoded_df], axis=1)

#### Encoding Gender

In [ ]:
df['is_female'] = df['gender'].apply(lambda x: 1 if x == 'F' else 0)

categorical_features.append('is_female')

#### All features & response 

In [ ]:
all_features = continuous_features + categorical_features
response = 'engaged'

In [ ]:
sample_df = df[all_features + [response]]
sample_df.columns = [x.replace(' ', '.') for x in sample_df.columns]
all_features = [x.replace(' ', '.') for x in all_features]

In [ ]:
sample_df.head()

# Training & Testing

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(sample_df[all_features], sample_df[response], test_size=0.3)

In [ ]:
sample_df.shape

In [ ]:
x_train.shape

In [ ]:
x_test.shape

## Building RandomForest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5
)

In [ ]:
rf_model.fit(X=x_train, y=y_train)

#### - Feature Importances

In [ ]:
rf_model.feature_importances_

In [ ]:
feature_importance_df = pd.DataFrame(list(zip(rf_model.feature_importances_, all_features)))
feature_importance_df.columns = ['feature.importance', 'feature']

feature_importance_df.sort_values(by='feature.importance', ascending=False)

#### - SHAP (SHapley Additive exPlanations) Analysis

In [ ]:
import shap
import matplotlib.pyplot as plt

# สร้าง TreeExplainer สำหรับ Random Forest (ทำงานเร็วสำหรับ tree-based models)
explainer = shap.TreeExplainer(rf_model)

# คำนวณ SHAP values สำหรับ test set
shap_values = explainer.shap_values(x_test)

In [ ]:
# Summary Plot - ตัวแปรไหนส่งผลต่อผลทำนายมากที่สุด
# (แสดงว่า feature ไหนมี impact สูงสุด และค่าสูง/ต่ำ มีผลบวก/ลบ)

# รองรับ SHAP output แบบใหม่ที่คืนค่าเป็น ndarray 3 มิติ: (n_samples, n_features, n_classes)
if hasattr(shap_values, "ndim") and shap_values.ndim == 3:
	shap_values = [shap_values[:, :, i] for i in range(shap_values.shape[2])]

shap.summary_plot(shap_values[1], x_test, plot_type="bar")

In [ ]:
# Bee Swarm Plot - ดูว่าแต่ละตัวแปรมี pattern ยังไง
# (สีแดง = ค่าสูง, สีน้ำเงิน = ค่าต่ำ)
shap.summary_plot(shap_values[1], x_test)

In [ ]:
# Dependence Plot - ความสัมพันธ์ระหว่างตัวแปรกับผลทำนาย
# ตัวอย่าง: top 3 important features
top_3_features = feature_importance_df.nlargest(3, 'feature.importance')['feature'].tolist()

for feature in top_3_features:
    feature_idx = all_features.index(feature)
    shap.dependence_plot(feature_idx, shap_values[1], x_test, feature_names=all_features)

In [ ]:
# Force Plot (Waterfall) - ดูว่า "ทำไมลูกค้าคนที่ 0 ถึงได้ผลลัพธ์แบบนั้น"
# แสดง baseline -> ปรับโดยตัวแปรต่างๆ -> ผลลัพธ์สุดท้าย
# ใช้ matplotlib=True เพื่อให้แสดงผลได้แม้ Javascript ถูกปิด/ยังไม่ trusted
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][0],
    x_test.iloc[0],
    feature_names=all_features,
    matplotlib=True
 )

In [ ]:
# ดูลูกค้าที่มีโอกาส engaged สูงที่สุดในชุดทดสอบ
engage_proba = rf_model.predict_proba(x_test)[:, 1]
target_idx = int(engage_proba.argmax())

print(f"customer index in x_test: {target_idx}")
print(f"predicted P(engaged)= {engage_proba[target_idx]:.4f}")

# รองรับทั้งกรณี expected_value เป็น list/array หรือ scalar
base_value = explainer.expected_value[1] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value

shap.force_plot(
    base_value,
    shap_values[1][target_idx],
    x_test.iloc[target_idx],
    feature_names=all_features,
    matplotlib=True
 )

In [ ]:
# Waterfall Plot - สำหรับลูกค้าที่ 0
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[1][0],
        base_values=explainer.expected_value[1] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value,
        data=x_test.iloc[0],
        feature_names=all_features
    )
)

In [ ]:
# Waterfall Plot - สำหรับลูกค้าที่มีโอกาส engaged สูงที่สุด
engage_proba = rf_model.predict_proba(x_test)[:, 1]
target_idx = int(engage_proba.argmax())
base_value = explainer.expected_value[1] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value

print(f"customer index in x_test: {target_idx}")
print(f"predicted P(engaged)= {engage_proba[target_idx]:.4f}")

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[1][target_idx],
        base_values=base_value,
        data=x_test.iloc[target_idx],
        feature_names=all_features
    )
)

## Evaluating Models

#### - Accuracy, Precision, and Recall

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [ ]:
in_sample_preds = rf_model.predict(x_train)
out_sample_preds = rf_model.predict(x_test)

In [ ]:
print('In-Sample Accuracy: %0.4f' % accuracy_score(y_train, in_sample_preds))
print('Out-of-Sample Accuracy: %0.4f' % accuracy_score(y_test, out_sample_preds))

In [ ]:
print('In-Sample Precision: %0.4f' % precision_score(y_train, in_sample_preds))
print('Out-of-Sample Precision: %0.4f' % precision_score(y_test, out_sample_preds))

In [ ]:
print('In-Sample Recall: %0.4f' % recall_score(y_train, in_sample_preds))
print('Out-of-Sample Recall: %0.4f' % recall_score(y_test, out_sample_preds))

#### - ROC & AUC

In [ ]:
from sklearn.metrics import roc_curve, auc

In [ ]:
in_sample_preds = rf_model.predict_proba(x_train)[:,1]
out_sample_preds = rf_model.predict_proba(x_test)[:,1]

In [ ]:
in_sample_fpr, in_sample_tpr, in_sample_thresholds = roc_curve(y_train, in_sample_preds)
out_sample_fpr, out_sample_tpr, out_sample_thresholds = roc_curve(y_test, out_sample_preds)

In [ ]:
in_sample_roc_auc = auc(in_sample_fpr, in_sample_tpr)
out_sample_roc_auc = auc(out_sample_fpr, out_sample_tpr)

print('In-Sample AUC: %0.4f' % in_sample_roc_auc)
print('Out-Sample AUC: %0.4f' % out_sample_roc_auc)

In [ ]:
import plotly.graph_objects as go

# Ensure ROC curve variables exist even if previous ROC cells were not run
if 'out_sample_fpr' not in globals() or 'out_sample_tpr' not in globals() or 'out_sample_roc_auc' not in globals():
    out_scores = out_sample_proba if 'out_sample_proba' in globals() else rf_model.predict_proba(x_test)[:, 1]
    out_sample_fpr, out_sample_tpr, _ = roc_curve(y_test, out_scores)
    out_sample_roc_auc = auc(out_sample_fpr, out_sample_tpr)

if 'in_sample_fpr' not in globals() or 'in_sample_tpr' not in globals() or 'in_sample_roc_auc' not in globals():
    in_scores = in_sample_proba if 'in_sample_proba' in globals() else rf_model.predict_proba(x_train)[:, 1]
    in_sample_fpr, in_sample_tpr, _ = roc_curve(y_train, in_scores)
    in_sample_roc_auc = auc(in_sample_fpr, in_sample_tpr)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=out_sample_fpr,
        y=out_sample_tpr,
        mode='lines',
        line=dict(color='darkorange'),
        name='Out-Sample ROC curve (area = %0.4f)' % out_sample_roc_auc
    )
)

fig.add_trace(
    go.Scatter(
        x=in_sample_fpr,
        y=in_sample_tpr,
        mode='lines',
        line=dict(color='navy'),
        name='In-Sample ROC curve (area = %0.4f)' % in_sample_roc_auc
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode='lines',
        line=dict(color='gray', width=1, dash='dash'),
        name='Random baseline'
    )
)

fig.update_layout(
    template='ggplot2',
    width=1000,
    height=700,
    title='RandomForest Model ROC Curve',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0.0, 1.0]),
    yaxis=dict(range=[0.0, 1.05]),
    legend=dict(x=1, y=0, xanchor='right', yanchor='bottom')
)

fig.show()

In [ ]:
# Executive metrics snapshot
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import pandas as pd

in_sample_class = rf_model.predict(x_train)
out_sample_class = rf_model.predict(x_test)
in_sample_proba = rf_model.predict_proba(x_train)[:, 1]
out_sample_proba = rf_model.predict_proba(x_test)[:, 1]

summary = {
    'n_rows': int(sample_df.shape[0]),
    'n_features': int(len(all_features)),
    'train_rows': int(x_train.shape[0]),
    'test_rows': int(x_test.shape[0]),
    'engagement_rate': float(sample_df[response].mean()),
    'in_accuracy': float(accuracy_score(y_train, in_sample_class)),
    'out_accuracy': float(accuracy_score(y_test, out_sample_class)),
    'in_precision': float(precision_score(y_train, in_sample_class)),
    'out_precision': float(precision_score(y_test, out_sample_class)),
    'in_recall': float(recall_score(y_train, in_sample_class)),
    'out_recall': float(recall_score(y_test, out_sample_class)),
    'in_auc': float(roc_auc_score(y_train, in_sample_proba)),
    'out_auc': float(roc_auc_score(y_test, out_sample_proba)),
}

top10 = feature_importance_df.sort_values('feature.importance', ascending=False).head(10).copy()
top10['feature.importance'] = top10['feature.importance'].round(4)

print('=== Model Snapshot ===')
for k, v in summary.items():
    if isinstance(v, float):
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: {v}')

print('\n=== Top 10 Features by RF Importance ===')
print(top10.to_string(index=False))

## Executive Deep-Dive: Engagement Prediction (สำหรับผู้บริหาร)

### 1) Executive Summary
โมเดล Random Forest สามารถแยกกลุ่มลูกค้าที่มีแนวโน้ม engage ได้ในระดับใช้งานจริง โดยมี Out-of-sample AUC = **0.8379** และ Out-of-sample Accuracy = **0.8709** บนข้อมูลทดสอบ 2,741 ราย (จากทั้งหมด 9,134 ราย)
อย่างไรก็ดี โครงสร้างข้อมูลมี class imbalance สูง (อัตรา engage เพียง **14.32%**) ทำให้โมเดลใน threshold ปัจจุบันมี **Recall ต่ำ (0.0974)** แม้ **Precision สูง (0.9500)** กล่าวคือ เมื่อโมเดลทำนายว่า "จะ engage" มักถูกต้อง แต่ยังพลาดลูกค้าที่มีโอกาส engage จำนวนมาก

### 2) Strategic Interpretation (Business Meaning)
- จุดแข็ง: โมเดลเหมาะกับ use case ที่เน้นความแม่นยำของรายชื่อลูกค้าเป้าหมาย (high-confidence targeting) เพื่อลดต้นทุนการยิงแคมเปญผิดกลุ่ม
- จุดอ่อน: สำหรับเป้าหมายด้าน "การขยายยอด conversion" เพียงอย่างเดียว threshold ปัจจุบันอาจ conservative เกินไป เพราะเก็บโอกาสลูกค้าได้น้อย
- ข้อสรุปเชิงผู้บริหาร: โมเดลนี้พร้อมใช้เป็น **Decision Support** ทันที แต่ควรปรับ threshold/segment strategy ให้สอดคล้องกับวัตถุประสงค์รายไตรมาส (Efficiency vs Growth)

### 3) Key Drivers ที่ส่งผลต่อการ engage สูงสุด
จาก Feature Importance อันดับต้น พบตัวขับเคลื่อนหลัก ได้แก่
1. `employment_status.Retired` (0.2901)
2. `total_claim_amount` (0.0796)
3. `income` (0.0772)
4. `customer_lifetime_value` (0.0660)
5. `monthly_premium_auto` (0.0596)
6. `months_since_policy_inception` (0.0548)
7. `sales_channel.Agent` (0.0420)
8. `marital_status.Divorced` (0.0399)
9. `months_since_last_claim` (0.0331)
10. `employment_status.Unemployed` (0.0295)

**นัยเชิงธุรกิจ:** ปัจจัยด้านสถานะอาชีพ, มูลค่าลูกค้า, รายได้, พฤติกรรมเคลม และช่องทางขาย มีอิทธิพลสูงต่อโอกาส engage จึงควรเป็นแกนหลักของ segmentation และข้อเสนอเฉพาะบุคคล

### 4) Risk & Governance ที่ผู้บริหารควรรับทราบ
- ความเสี่ยงด้านโอกาสสูญหาย (Opportunity Loss): Recall ต่ำอาจทำให้พลาดลูกค้ากลุ่มที่ convert ได้จริง
- ความเสี่ยงด้าน fairness/compliance: การใช้ตัวแปรเชิงประชากร (เช่นสถานะการจ้างงาน/สถานภาพสมรส) ต้องอยู่ภายใต้นโยบายการใช้งานข้อมูลที่เหมาะสม
- ความเสี่ยงด้าน drift: พฤติกรรมลูกค้าและช่องทางขายเปลี่ยนเร็ว จำเป็นต้องติดตาม model performance เป็นรอบเวลา

### 5) Recommended Action Plan (90 วัน)
1. **Threshold Optimization by Objective**: ทดสอบ threshold หลายระดับเพื่อหา operating point ที่เหมาะกับ KPI (เช่น maximize expected profit แทน accuracy)
2. **Two-Tier Campaign Design**:
   - Tier A (High precision): ส่งข้อเสนอ premium ให้กลุ่มคะแนนสูงสุด
   - Tier B (Growth capture): ขยายฐานด้วยข้อเสนอเบากว่าเพื่อเพิ่ม recall
3. **Uplift/A-B Validation**: วัด incremental lift เทียบกับวิธีเลือกกลุ่มแบบเดิม
4. **Monitoring Dashboard**: ติดตาม AUC, Precision@K, Recall@K, Conversion lift รายสัปดาห์
5. **Governance Review**: ตรวจสอบการใช้ feature ที่อ่อนไหวและกำหนด policy การใช้งาน

### 6) KPI Targets ที่แนะนำสำหรับไตรมาสถัดไป
- เพิ่ม Out-of-sample Recall อย่างน้อย **2x** จาก baseline (โดยรักษา Precision ให้อยู่ระดับที่ยอมรับได้ทางธุรกิจ)
- เพิ่ม Conversion Lift ของแคมเปญแบบ model-driven เทียบ control
- ลด Cost per Engaged Customer ผ่านการจัดลำดับกลุ่มเป้าหมายตาม propensity score

### 7) Final Executive Takeaway
โมเดลปัจจุบันมีศักยภาพสูงในการจัดลำดับความสำคัญลูกค้าและลดการใช้ทรัพยากรผิดกลุ่ม แต่เพื่อปลดล็อกมูลค่าทางธุรกิจสูงสุด ควรเปลี่ยนจากแนวคิด "โมเดลแม่นหรือไม่" ไปสู่ "โมเดลสร้างกำไรเพิ่มได้เท่าไร" ผ่านการปรับ threshold, ออกแบบแคมเปญสองชั้น และวัดผลแบบ uplift อย่างมีวินัย